# 09 — Parquet path dataset со множеством файлов и эволюцией схемы

До сих пор все output'ы шли через `saveAsTable` → Hive-managed таблицы → namespace `hadoop-cluster` + имя `default.<table>`.
Здесь делаем по-другому: пишем напрямую в HDFS как Parquet (path-based dataset). OpenLineage отметит это как датасет с:
- **namespace** = `hdfs://namenode:9000` (URI scheme + authority)
- **name** = путь, `/tmp/lineage/events_parquet`

Source — **`default.raw_orders`** (плюс `raw_customers` для batch 2). Это важно: если писать `spark.range(...)`, OL-Spark не сможет привязать колонки к upstream dataset'у (`InputFieldsCollector: Could not extract dataset identifier from Range`) и column lineage facet будет пустым в любой версии OL. Чтобы col-lineage реально появился — source должен быть tracked dataset.

В тот же путь подряд делаем три **append**'а с разной схемой:

| Batch | Source                       | Колонки                                  | Файлов добавляется |
|-------|------------------------------|------------------------------------------|--------------------|
| 1     | `raw_orders`                 | `event_id, user_id, ts`                  | 8                  |
| 2     | `raw_orders ⨝ raw_customers` | `event_id, user_id, ts, country`         | 8 (+1 колонка)     |
| 3     | `raw_orders`                 | `event_id, ts, device`                   | 8 (drop `user_id`+`country`, новая `device`) |

Итого: **24 parquet-файла в одной директории, 3 разных on-disk схемы**. В Marquez API увидит **3 schema versions** одного path-based датасета + реальный column lineage от `raw_orders`/`raw_customers` к каждой версии.

Prerequisite: прогнан `00_setup.ipynb` (нужны `raw_orders`, `raw_customers`). **Restart Jupyter kernel** перед запуском.

In [1]:
try:
    spark.stop()
except NameError:
    pass

from pyspark.sql import SparkSession, functions as F
spark = (
    SparkSession.builder
    .appName('lineage_09_parquet_schema_drift')
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.appName == 'lineage_09_parquet_schema_drift', \
    f'appName fallback to {spark.sparkContext.appName!r} — restart Jupyter kernel'
print('app name:', spark.sparkContext.appName)

PATH = 'hdfs://namenode:9000/tmp/lineage/events_parquet'

# Чистим путь, чтобы старые файлы не путали схемы.
_FS  = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
_Path = spark._jvm.org.apache.hadoop.fs.Path
_FS.delete(_Path(PATH), True)
print('path cleaned:', PATH)

def list_parquet_files(path):
    p = _Path(path)
    if not _FS.exists(p):
        return []
    return [s.getPath().getName() for s in _FS.listStatus(p) if s.getPath().getName().endswith('.parquet')]

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 13:15:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/27 13:15:15 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/05/27 13:15:24 WARN BlackbirdModule: Unable to find Java 9+ MethodHandles.privateLookupIn.  Blackbird is not performing optimally!
26/05/27 13:15:24 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.
26/05/27 13:15:24 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.


app name: lineage_09_parquet_schema_drift
path cleaned: hdfs://namenode:9000/tmp/lineage/events_parquet


## Batch 1: initial schema `(event_id, user_id, ts)` → 8 файлов

`mode('overwrite')` — пишем с нуля. `repartition(8)` даёт ровно 8 parquet-частей.

In [2]:
batch1 = spark.sql('''
    SELECT
        order_id    AS event_id,
        customer_id AS user_id,
        order_ts    AS ts
    FROM default.raw_orders
    WHERE order_id < 200
''').repartition(8)
batch1.write.mode('overwrite').parquet(PATH)
print('schema written:'); batch1.printSchema()
print(f'rows written: {batch1.count()}, files now: {len(list_parquet_files(PATH))}')

26/05/27 13:15:30 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist
26/05/27 13:15:30 WARN HiveConf: HiveConf of name hive.log.dir does not exist
26/05/27 13:15:30 WARN HiveConf: HiveConf of name hive.root.logger does not exist
26/05/27 13:15:30 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic
                                                                                

schema written:
root
 |-- event_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- ts: timestamp (nullable = true)

rows written: 200, files now: 8


## Batch 2: append, добавляем колонку `country` → +8 файлов

Parquet allow'ит писать файлы с разной schema в одну директорию — у каждого файла собственный schema в footer. На чтении без `mergeSchema` Spark возьмёт схему из одного файла; с `mergeSchema=true` — объединит.

In [3]:
batch2 = spark.sql('''
    SELECT
        o.order_id    AS event_id,
        o.customer_id AS user_id,
        o.order_ts    AS ts,
        c.country     AS country
    FROM default.raw_orders o
    JOIN default.raw_customers c ON c.id = o.customer_id
    WHERE o.order_id BETWEEN 200 AND 399
''').repartition(8)
batch2.write.mode('append').parquet(PATH)
print('schema written:'); batch2.printSchema()
print(f'rows written: {batch2.count()}, files now: {len(list_parquet_files(PATH))}')

schema written:
root
 |-- event_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- country: string (nullable = true)

rows written: 200, files now: 16


## Batch 3: append, drop `user_id` и `country`, добавляем `device` → +8 файлов

Здесь жёсткий schema drift: пропадают сразу 2 колонки, появляется новая. На последующем чтении с `mergeSchema=true` пропавшие у этих файлов поля будут `NULL`.

In [4]:
batch3 = spark.sql('''
    SELECT
        order_id  AS event_id,
        order_ts  AS ts,
        element_at(array('ios', 'android', 'web'), cast(order_id % 3 + 1 AS INT)) AS device
    FROM default.raw_orders
    WHERE order_id >= 400
''').repartition(8)
batch3.write.mode('append').parquet(PATH)
print('schema written:'); batch3.printSchema()
print(f'rows written: {batch3.count()}, files now: {len(list_parquet_files(PATH))}')

schema written:
root
 |-- event_id: long (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- device: string (nullable = true)

rows written: 100, files now: 24


## Чтение всех файлов с `mergeSchema=true`

Объединённая schema = union трёх batch'ей: `(event_id, user_id, ts, country, device)`. У строк batch1 будут NULL в `country` и `device`, у batch3 — NULL в `user_id` и `country`.

In [5]:
merged = spark.read.option('mergeSchema', 'true').parquet(PATH)
print('union schema:'); merged.printSchema()
print(f'total rows: {merged.count()}, total files: {len(merged.inputFiles())}')

# Сводка NULL'ов по колонкам — наглядная картина schema drift.
merged.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ['event_id', 'user_id', 'ts', 'country', 'device']
]).show(truncate=False)

union schema:
root
 |-- event_id: long (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- device: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- country: string (nullable = true)

total rows: 500, total files: 24
+--------+-------+---+-------+------+
|event_id|user_id|ts |country|device|
+--------+-------+---+-------+------+
|0       |100    |0  |300    |400   |
+--------+-------+---+-------+------+



In [6]:
spark.stop()

## Что смотреть в Marquez

### Dataset
В Marquez UI namespace будет **не** `hadoop-cluster`, а `hdfs://namenode:9000`. Найти:

```bash
curl -s 'http://localhost:5000/api/v1/namespaces' | jq '.namespaces[].name'
curl -s 'http://localhost:5000/api/v1/namespaces/hdfs%3A%2F%2Fnamenode%3A9000/datasets' | jq '.datasets[].name'
```

Должен появиться датасет с именем `/tmp/lineage/events_parquet`.

### Schema versions
Каждый из 3 write'ов = новая version у этого dataset'а. Три разных набора `fields`:

```bash
NS='hdfs%3A%2F%2Fnamenode%3A9000'
DS='%2Ftmp%2Flineage%2Fevents_parquet'
curl -s "http://localhost:5000/api/v1/namespaces/$NS/datasets/$DS/versions" \
  | jq '.versions[].fields | map(.name)'
```

Должно вернуть три массива:
```
["event_id", "user_id", "ts"]
["event_id", "user_id", "ts", "country"]
["event_id", "ts", "device"]
```

### Column lineage
Поскольку source — реальная Hive-таблица `raw_orders` (а не synthetic `spark.range`), column lineage facet будет заполнен:

```bash
curl -s "http://localhost:5000/api/v1/namespaces/$NS/datasets/$DS/versions" \
  | jq '.versions[0].facets.columnLineage'
```

Ожидаемые edges (для batch1, последняя version):
- `event_id ← raw_orders.order_id` (subtype `IDENTITY`)
- `user_id  ← raw_orders.customer_id` (subtype `IDENTITY`)
- `ts       ← raw_orders.order_ts` (subtype `IDENTITY`)

В batch2 + `country ← raw_customers.country`. В batch3 — `device` без upstream DIRECT (вычислено из литерала и `order_id`).

> ℹ️ Cluster использует `spark.openlineage.columnLineage.datasetLineageEnabled=true` (см. `spark-defaults.conf`) — facet может прийти в форме `dataset[]` вместо `fields[]`. Marquez поддерживает оба формата.

### Application-level vs batch-level jobs
Под appName `lineage_09_parquet_schema_drift` Marquez создаст **APPLICATION-level placeholder** job с тем же именем — у него `inputs=[]`, `outputs=[]`, `latestRun=null`, это нормально (OL-Spark 1.x так помечает корень). Реальные runs живут под именами `unknown.lineage_09_parquet_schema_drift.<node>.<output>` (префикс `unknown` — race condition в OL-Spark с резолвом appName на старте listener'а; косметика, не влияет на lineage).

### Когда полезно сравнивать с Hive (05)
В `05_schema_versions` каждая запись делает `mode('overwrite').saveAsTable` → Hive под капотом дропает таблицу и создаёт заново. У path-based parquet здесь подход обратный: **append'ом аккумулируем** файлы с разными схемами в одной директории, никаких DROP. Marquez одинаково регистрирует обе истории как schema versions, но семантика на стороне Spark разная — это полезно понимать.

## Cleanup

Когда захочешь стереть данные с HDFS:

```python
_FS.delete(_Path('hdfs://namenode:9000/tmp/lineage/events_parquet'), True)
```

Или из shell контейнера:
```bash
docker compose exec namenode hdfs dfs -rm -r -skipTrash /tmp/lineage/events_parquet
```